# 13과 - 에이전트 메모리


## 설정

이 노트북에서는 **Microsoft Agent Framework**(MAF)를 사용하여 **지속적인 메모리**를 가진 여행 예약 에이전트를 만드는 방법을 보여줍니다.

작업 메모리, 단기 메모리, 장기 메모리 등 다양한 유형의 에이전트 메모리가 대화 전반에 걸쳐 에이전트가 정보를 유지하고 사용하는 방식에 어떤 영향을 미치는지 배우게 됩니다.

**필수 조건:**
- 배포된 챗 모델(e.g. `gpt-4o-mini`)이 포함된 Azure AI Foundry 프로젝트.
- Azure CLI에 로그인 — 터미널에서 `az login` 명령어 실행.
- `AZURE_AI_PROJECT_ENDPOINT` — Azure AI Foundry 프로젝트 엔드포인트.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — 배포된 모델의 이름.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## 에이전트 메모리 유형

AI 에이전트는 각각 다른 목적을 가진 다양한 종류의 메모리를 활용할 수 있습니다:

### 작업 메모리
대화 스레드 자체 — 단일 세션에서 주고받은 메시지들. 에이전트는 동일 스레드 내 이전 메시지를 참조하여 일관성을 유지할 수 있습니다. MAF에서는 **`agent.create_session()`** 을 통해 생성되며, `AgentSession`을 반환합니다.

### 단기 메모리
작업 또는 세션이 지속되는 동안 유지되지만 영구적으로 저장되지 않는 정보입니다. 예를 들어, 에이전트는 다중 턴 계획 대화 중에 사실을 축적하고 이를 사용해 최종 일정을 생성할 수 있습니다.

### 장기 메모리
**세션 간에** 지속되는 선호도와 사실들입니다. 재방문 사용자는 식이 제한이나 여행 스타일을 반복할 필요가 없어야 합니다. 장기 메모리는 일반적으로 외부 저장소 — 데이터베이스, 파일, 또는 벡터 인덱스 — 에 의해 지원되며 도구를 통해 에이전트에 제공됩니다.


## 세션을 통한 작업 기억

가장 간단한 형태의 기억은 대화 세션입니다. 동일한 세션(`agent.create_session()`을 통해 생성된)을 연속적인 `agent.run()` 호출에 전달하면 에이전트는 해당 대화의 전체 기록을 보고 이전 세부 정보를 기억할 수 있습니다.

여행 에이전트를 만들고 작업 기억을 시연해 보겠습니다.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

에이전트는 두 메시지가 동일한 세션을 공유하기 때문에 예산을 올바르게 기억했습니다. 이것이 바로 **작동 기억(working memory)**이며 — 세션이 지속되는 동안에만 존재합니다.

### 새 스레드에서는 어떻게 될까요?

**새로운** 세션을 만들면 에이전트는 이전 대화에 대한 기억이 없습니다:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## 장기 기억 패턴

사용자 선호도를 **세션 간에** 기억하려면, 대화 스레드 외부에 존재하는 지속적인 저장소가 필요합니다. 에이전트는 이 저장소에 접근하기 위해 정보를 저장하고 검색할 수 있는 **도구**—호출 가능한 함수—를 사용합니다.

아래에서는 간단한 메모리 내 선호도 저장소를 구현합니다(실제 환경에서는 데이터베이스나 벡터 인덱스를 백엔드로 사용). 이 저장소를 에이전트가 사용할 수 있는 도구로 노출합니다.

### 아키텍처
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### 시나리오 1 — 첫 방문 사용자가 기념일 여행 예약하기

Sarah가 처음 방문합니다. 상담원은 도구를 통해 그녀의 선호도를 저장하고 이를 사용하여 호텔을 추천해야 합니다.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### 시나리오 2 — 사라가 몇 주 후에 돌아옴

사라는 **새로운 스레드**를 시작합니다(새 세션을 시뮬레이션). 작업 메모리는 비어 있지만, 장기 선호 저장소에는 여전히 그녀의 정보가 있습니다. 에이전트는 이를 검색하여 개인화된 추천에 사용해야 합니다.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## 요약

이번 수업에서는 세 가지 유형의 에이전트 메모리와 마이크로소프트 에이전트 프레임워크를 사용하여 이를 구현하는 방법을 배웠습니다:

| 메모리 유형 | MAF 메커니즘 | 수명 |
|---|---|---|
| **작업용** | `agent.create_session()` | 단일 대화 |
| **단기** | 스레드 내 누적된 컨텍스트 | 단일 작업 / 세션 |
| **장기** | `@tool` 함수를 통해 액세스하는 외부 저장소 | 세션 간 |

### 핵심 포인트
1. **`agent.create_session()`** 은 작업용 메모리를 제공합니다 — 에이전트는 세션 내에서 전체 대화 이력을 볼 수 있습니다.
2. **새 세션에서는 컨텍스트가 사라집니다** — 장기 메모리가 없으면 에이전트는 과거 대화를 기억할 수 없습니다.
3. **`@tool` 함수** 가 그 격차를 메웁니다 — 에이전트가 정보를 지속 저장소에 저장하고 불러올 수 있게 합니다.
4. **개인화는 시간이 지날수록 향상됩니다** — 저장된 선호도가 많을수록 에이전트의 추천이 좋아집니다.

### 실제 응용 사례
- **고객 서비스**: 고객 이력과 선호도 기억
- **개인 비서**: 수일 또는 수주에 걸쳐 컨텍스트 유지
- **의료**: 환자 정보와 선호도 추적
- **전자 상거래**: 이력을 기반으로 한 맞춤형 쇼핑

### 다음 단계
- 메모리 딕셔너리를 데이터베이스 또는 벡터 스토어(e.g. Azure AI Search)로 대체
- 시간 민감 정보에 대해 메모리 만료 추가
- 공유 메모리를 가진 다중 에이전트 시스템 구축
- 지식 그래프 기반 메모리를 위한 Cognee 노트북 탐색


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책 조항**:  
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 정확성을 위해 최선을 다하고 있으나, 자동 번역에는 오류나 부정확한 내용이 포함될 수 있음을 유의하시기 바랍니다. 원문 문서는 해당 언어의 권위 있는 출처로 간주되어야 합니다. 중요한 정보에 대해서는 전문적인 인간 번역을 권장합니다. 이 번역물의 사용으로 발생하는 오해나 잘못된 해석에 대해서는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
